In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
from mlxtend.frequent_patterns import apriori, association_rules

uploaded = files.upload()
file_name = next(iter(uploaded))

df = pd.read_csv(file_name)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:")
display(df.head())

print("\nMissing Values:")
print(df.isnull().sum().sum())

Saving BookMyShow_Weka_Classification_Final.csv to BookMyShow_Weka_Classification_Final (1).csv
Dataset Shape: (20000, 14)

First 5 Records:


,total_tickets,ticket_price,total_amount,available_seats,payment_method,booking_hour,booking_day,movie_genre,movie_language,movie_rating,show_type,booking_channel,duration,transaction_status
0,3,363.67,1091.01,179,Wallet,16,5,Action,Hindi,6.8,Regular,Kiosk,127,Success
1,1,364.32,364.32,208,Credit Card,16,3,Romance,Telugu,9.3,Regular,Mobile App,123,Success
2,2,187.61,375.22,58,UPI,19,5,Sci-Fi,Bengali,6.7,3D,Mobile App,194,Success
3,6,120.11,720.66,95,UPI,0,5,Action,Kannada,6.8,Regular,Website,187,Success
4,1,371.18,371.18,142,Credit Card,18,4,Drama,Bengali,8.0,IMAX,Website,151,Success



Missing Values:
0


In [ ]:
data = df.drop(
    columns=["ticket_price", "total_amount"],
    errors="ignore"
).copy()

data = data.drop_duplicates().reset_index(drop=True)

data["total_tickets"] = data["total_tickets"].astype(str)
data["booking_day"] = data["booking_day"].astype(str)
data["booking_hour"] = data["booking_hour"].astype(str)

data["available_seats"] = pd.cut(
    data["available_seats"],
    bins=[-np.inf, 50, 100, 200, np.inf],
    labels=["Low", "Medium", "High", "Very High"]
).astype(str)

data["movie_rating"] = pd.cut(
    data["movie_rating"],
    bins=[-np.inf, 4, 6, 8, np.inf],
    labels=["Low", "Average", "Good", "High"]
).astype(str)

data["duration"] = pd.cut(
    data["duration"],
    bins=[-np.inf, 90, 120, 150, np.inf],
    labels=["Short", "Medium", "Long", "Very Long"]
).astype(str)

data = data.replace("nan", np.nan).dropna()

print("Preprocessed Shape:", data.shape)
display(data.head())

Preprocessed Shape: (20000, 12)


,total_tickets,available_seats,payment_method,booking_hour,booking_day,movie_genre,movie_language,movie_rating,show_type,booking_channel,duration,transaction_status
0,3,High,Wallet,16,5,Action,Hindi,Good,Regular,Kiosk,Long,Success
1,1,Very High,Credit Card,16,3,Romance,Telugu,High,Regular,Mobile App,Long,Success
2,2,Medium,UPI,19,5,Sci-Fi,Bengali,Good,3D,Mobile App,Very Long,Success
3,6,Medium,UPI,0,5,Action,Kannada,Good,Regular,Website,Very Long,Success
4,1,High,Credit Card,18,4,Drama,Bengali,Good,IMAX,Website,Very Long,Success


In [ ]:
transactions = pd.get_dummies(
    data.astype(str),
    prefix_sep="="
).astype(bool)

print("Transaction Matrix Shape:", transactions.shape)
display(transactions.head())

Transaction Matrix Shape: (20000, 79)


,total_tickets=1,total_tickets=2,total_tickets=3,total_tickets=4,total_tickets=5,total_tickets=6,available_seats=High,available_seats=Low,available_seats=Medium,available_seats=Very High,...,booking_channel=Mobile App,booking_channel=Partner App,booking_channel=Website,duration=Long,duration=Medium,duration=Short,duration=Very Long,transaction_status=Failed,transaction_status=Refunded,transaction_status=Success
0,False,False,True,False,False,False,True,False,False,False,...,False,False,False,True,False,False,False,False,False,True
1,True,False,False,False,False,False,False,False,False,True,...,True,False,False,True,False,False,False,False,False,True
2,False,True,False,False,False,False,False,False,True,False,...,True,False,False,False,False,False,True,False,False,True
3,False,False,False,False,False,True,False,False,True,False,...,False,False,True,False,False,False,True,False,False,True
4,True,False,False,False,False,False,True,False,False,False,...,False,False,True,False,False,False,True,False,False,True


In [ ]:
frequent_itemsets = apriori(
    transactions,
    min_support=0.05,
    use_colnames=True,
    max_len=3
)

frequent_itemsets["length"] = (
    frequent_itemsets["itemsets"].apply(len)
)

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
).reset_index(drop=True)

print("Frequent Itemsets:")
display(frequent_itemsets.head(20))

Frequent Itemsets:


,support,itemsets,length
0,0.66920,(transaction_status=Success),1
1,0.59650,(show_type=Regular),1
2,0.54265,(available_seats=High),1
3,0.53130,(movie_rating=Good),1
4,0.47940,(booking_channel=Mobile App),1
5,0.42050,(duration=Very Long),1
6,0.41565,"(show_type=Regular, transaction_status=Success)",2
7,0.39085,"(available_seats=High, transaction_status=Succ...",2
8,0.38645,(duration=Long),1
9,0.37600,"(movie_rating=Good, transaction_status=Success)",2


In [ ]:
if frequent_itemsets.empty:
    print("No frequent itemsets found. Reduce the minimum support.")
else:
    rules = association_rules(
        frequent_itemsets,
        metric="confidence",
        min_threshold=0.30
    )

    rules = rules[rules["lift"] >= 1].copy()

    rules = rules.sort_values(
        by=["confidence", "lift"],
        ascending=False
    )

    if rules.empty:
        print("No rules met the selected confidence and lift thresholds.")
    else:
        rules["antecedents"] = rules["antecedents"].apply(
            lambda x: ", ".join(sorted(x))
        )
        rules["consequents"] = rules["consequents"].apply(
            lambda x: ", ".join(sorted(x))
        )

        display(
            rules[
                ["antecedents", "consequents",
                 "support", "confidence", "lift"]
            ].head(20).round(4)
        )

,antecedents,consequents,support,confidence,lift
783,"payment_method=UPI, total_tickets=1",transaction_status=Success,0.0680,0.8878,1.3267
561,"payment_method=UPI, total_tickets=2",transaction_status=Success,0.0816,0.8826,1.3189
1203,"payment_method=Credit Card, total_tickets=1",transaction_status=Success,0.0526,0.8768,1.3102
830,"payment_method=Credit Card, total_tickets=2",transaction_status=Success,0.0656,0.8574,1.2813
183,"available_seats=High, payment_method=UPI",transaction_status=Success,0.1403,0.8544,1.2768
279,"booking_channel=Mobile App, payment_method=UPI",transaction_status=Success,0.1182,0.8518,1.2729
676,"available_seats=Medium, payment_method=UPI",transaction_status=Success,0.0748,0.8442,1.2614
607,"booking_channel=Website, payment_method=UPI",transaction_status=Success,0.0780,0.8420,1.2582
789,"movie_rating=Average, payment_method=UPI",transaction_status=Success,0.0676,0.8356,1.2487
159,"payment_method=UPI, show_type=Regular",transaction_status=Success,0.1484,0.8344,1.2468
